# 01 First call 

This notebook demonstrates the first call to the document intelligence API.

In [1]:
ROOT = "../"
DATA = ROOT + "data/"
RECEIPTS = DATA + "receipts/"
OUTPUT = ROOT + "output/"

from pathlib import Path

Path(OUTPUT).mkdir(exist_ok=True)

## Configure endpoint and authenticate

In [4]:
import os
from dotenv import load_dotenv
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.core.credentials import AzureKeyCredential

load_dotenv()

endpoint = os.getenv("DOCUMENTINTELLIGENCE_ENDPOINT")
key = os.getenv("DOCUMENTINTELLIGENCE_API_KEY")

if not endpoint or not key:
    raise ValueError("Please set the DOCUMENTINTELLIGENCE_ENDPOINT and DOCUMENTINTELLIGENCE_API_KEY environment variables.")

client = DocumentIntelligenceClient(endpoint=endpoint, credential=AzureKeyCredential(key))

## Analyze receipt document and measure processing time

In [6]:
import time

start = time.time()
with open(RECEIPTS+"r04_e_baldi.pdf", "rb") as f:
    poller = client.begin_analyze_document(model_id='prebuilt-receipt', body=f)

result = poller.result()

print("Finished in ", f"{(time.time() - start):.1f}", "seconds. Status:", poller.status())

Finished in  4.3 seconds. Status: succeeded


## Inspect result

In [7]:
result

{'apiVersion': '2024-11-30', 'modelId': 'prebuilt-receipt', 'stringIndexType': 'textElements', 'content': 'E. BALDI\n375 North Canon Drive\nBeverly Hills, CA 90210\nTUE JANUARY 30,2018\nCHECK #278470-1\nTABLE #6\n1 ICED TEA\n$4.50\n1 ROMBO CARCIOFI FUNGH\n$48.00\n1 SPARKLING WATER LARG\n$8.50\n1 BRANZINO GRIGLIATO\n1 CARPACCIO RED SNAPPE\n$40.00\n$33.00\nFOOD\nSUB-TOTAL\n:\n$134.00\n:\n$134.00\nTAX\nTOTAL\n:\n$12.73\n$146.73\nTime: 14:22\n2 CUSTOMEDO', 'pages': [{'pageNumber': 1, 'angle': -0.15629999339580536, 'width': 8.2639, 'height': 11.6944, 'unit': 'inch', 'words': [{'content': 'E.', 'polygon': [2.2876, 1.1164, 2.8558, 1.0717, 2.8718, 1.4568, 2.3042, 1.4904], 'confidence': 0.992, 'span': {'offset': 0, 'length': 2}}, {'content': 'BALDI', 'polygon': [3.3046, 1.0436, 5.0407, 0.9037, 5.0429, 1.286, 3.3231, 1.4287], 'confidence': 0.995, 'span': {'offset': 3, 'length': 5}}, {'content': '375', 'polygon': [0.2834, 1.6323, 1.2737, 1.5917, 1.2841, 1.9839, 0.292, 2.0214], 'confidence': 0.998

In [13]:
print("Model:", result.model_id)
print("Pages: ", len(result.pages))
print("Documents", len(result.documents))

receipt = result.documents[0]
print("Document type: ", receipt.doc_type, "| Confidence:", receipt.confidence)
print("Fields:", list(receipt.fields.keys()))

Model: prebuilt-receipt
Pages:  1
Documents 1
Document type:  receipt.retailMeal | Confidence: 0.956
Fields: ['CountryRegion', 'Items', 'MerchantAddress', 'MerchantName', 'ReceiptType', 'Subtotal', 'Total', 'TotalTax', 'TransactionDate', 'TransactionTime']


In [18]:
for name in ['MerchantName', 'TransactionDate', 'Total', 'Subtotal', 'TotalTax']:
    field = receipt.fields.get(name)
    if field is None:
        print(f"{name}: Not found")
    elif field.type == "date":
        print(f"{name}: {field.value_date} | Confidence: {field.confidence}")
    elif field.type == "currency":
        print(f"{name}: {field.value_currency.amount} {field.value_currency.currency_code}| Confidence: {field.confidence}")
    else:
        print(f"{name}: {field.value_string} | Confidence: {field.confidence}")

MerchantName: E. BALDI | Confidence: 0.972
TransactionDate: 2018-01-30 | Confidence: 0.987
Total: 146.73 USD| Confidence: 0.985
Subtotal: 134.0 USD| Confidence: 0.986
TotalTax: 12.73 USD| Confidence: 0.992


In [21]:
# crea un dizionario semplice con i valori per type
fields_dict = {}

for name in ['MerchantName', 'TransactionDate', 'Total', 'Subtotal', 'TotalTax', 'Tip']:
    field = receipt.fields.get(name)
    if field is None:
        fields_dict[name] = None
    elif field.type == "date":
        fields_dict[name] = field.value_date
    elif field.type == "currency":
        fields_dict[name] = field.value_currency.amount
        fields_dict[name + "_currency"] = field.value_currency.currency_code
    else:
        fields_dict[name] = field.value_string

fields_dict

{'MerchantName': 'E. BALDI',
 'TransactionDate': datetime.date(2018, 1, 30),
 'Total': 146.73,
 'Total_currency': 'USD',
 'Subtotal': 134.0,
 'Subtotal_currency': 'USD',
 'TotalTax': 12.73,
 'TotalTax_currency': 'USD',
 'Tip': None}

In [23]:

import json


results = Path(DATA+"results/")
results.mkdir(exist_ok=True)

raw = result.as_dict()
out_file = results / "r04_e_baldi.pdf.prebuilt-receipt.json"

out_file.write_text(json.dumps(raw, indent=4))


85281